In [2]:
import pandas as pd

customers = pd.read_csv("../data/raw/customers.csv")
subscriptions = pd.read_csv("../data/raw/subscriptions.csv")
revenue = pd.read_csv("../data/raw/revenue.csv")

In [3]:
customers['signup_date'] = pd.to_datetime(customers['signup_date'])
customers['churn_date'] = pd.to_datetime(customers['churn_date'])

subscriptions['month'] = pd.to_datetime(subscriptions['month'])
revenue['month'] = pd.to_datetime(revenue['month'])


In [4]:
#First subscription month per cusomer
first_subscription =(
    subscriptions
    .groupby('customer_id')['month']
    .min()
    .reset_index(name ='first_subscription_month')
)
activation = customers.merge(
    first_subscription,
    on='customer_id',
    how='left'
)
activation['days_to_activate']=(
    activation['first_subscription_month'] - activation['signup_date']
).dt.days


In [5]:
# Convert each customer's signup date into a monthly period (e.g., 2023-05)
# This allows us to compare dates at the month level instead of exact timestamps.
customers['signup_month'] = customers['signup_date'].dt.to_period('M')


# If a customer has not churned, churn_month will be NaT (Not a Time).
customers['churn_month'] = customers['churn_date'].dt.to_period('M')

# Calculate the customer's lifetime in months.
# Subtracting two Period('M') objects gives a Period difference.
# The `.n` attribute extracts the numeric number of months from that difference.
# If churn_month is missing (customer still active), return None instead.
customers['lifetime_months'] = (
    customers['churn_month'] - customers['signup_month']
).apply(lambda x: x.n if pd.notnull(x) else None)

In [6]:
customers['lifetime_months'].describe()


count    168.000000
mean       5.315476
std        4.078775
min        0.000000
25%        2.000000
50%        4.000000
75%        8.000000
max       17.000000
Name: lifetime_months, dtype: float64

In [7]:
customers['lifetime_months'].value_counts().sort_index()


lifetime_months
0.0      7
1.0     15
2.0     29
3.0     26
4.0     13
5.0     14
6.0     10
7.0     10
8.0     10
9.0      7
10.0     2
11.0     5
12.0     6
13.0     4
14.0     5
15.0     3
16.0     1
17.0     1
Name: count, dtype: int64

In [8]:
# Build a retention-style table based on customer lifetimes.

retention = (
    customers
        # Remove customers who do not have a lifetime_months value.
        # These are typically active customers who haven't churned yet.
        .dropna(subset=['lifetime_months'])

        # Group customers by how many months they stayed before churning.
        # Example: all customers who churned after 3 months go into the same group.
        .groupby('lifetime_months')

        # Count how many customers are in each lifetime group.
        # This gives the number of customers who churned at each month.
        .size()

        # Convert the counts into a cumulative sum.
        # This shows how many customers have churned *up to* each lifetime month.
        # Example: if 5 churned at month 1 and 7 at month 2, cumsum = [5, 12].
        .cumsum()

        # Turn the result into a clean DataFrame with a readable column name.
        .reset_index(name='churned_customers')
)

retention

,lifetime_months,churned_customers
0,0.0,7
1,1.0,22
2,2.0,51
3,3.0,77
4,4.0,90
5,5.0,104
6,6.0,114
7,7.0,124
8,8.0,134
9,9.0,141


## Retention & Churn Insight
Customer churn is heavily front-loaded, with a median lifetime of approximately 4 months. Nearly half of all churned customers leave within the first three months, indicating that early post-activation experience is the primary driver of retention outcomes.
